In [1]:
import numpy as np
import pandas as pd
import glob
import os
import h5py
from mne.filter import filter_data, notch_filter
import gc as garbageCollector
from scipy.signal import fftconvolve
import matplotlib.pyplot as plt
%matplotlib qt
from mne.time_frequency import psd_array_multitaper
import scipy.io as sio
from scipy.signal import detrend
from tqdm import tqdm
from collections import Counter
from bisect import bisect

In [2]:
from datetime import datetime, time as datetime_time, timedelta

def time_diff(start, end):
    if isinstance(start, datetime_time): # convert to datetime
        assert isinstance(end, datetime_time)
        start, end = [datetime.combine(datetime.min, t) for t in [start, end]]
    if start <= end: # e.g., 10:33:26-11:15:49
        return end - start
    else: # end < start e.g., 23:55:00-00:25:00
        end += timedelta(1) # +day
        assert end > start
        return end - start

In [3]:
def get_grass_start_end_time(starttime_raw, endtime_raw):
    
    time_str_elements = starttime_raw.flatten()
    start_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))
    time_str_elements = endtime_raw.flatten()
    end_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))

    start_time = start_time.split(':')
    second_elements = start_time[-1].split('.')
    start_time = datetime.datetime(1990,1,1,hour=int(float(start_time[0])), minute=int(float(start_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))
    end_time = end_time.split(':')
    second_elements = end_time[-1].split('.')
    end_time = datetime.datetime(1990,1,1,hour=int(float(end_time[0])), minute=int(float(end_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))

    return start_time, end_time

def check_load_mgh_dataset(signal_path, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signal_path)
        data_path = os.path.basename(signal_path)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        if 'grass' in signal_path:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
#         if 'grass' in signal_path:
#             with h5py.File(label_path) as ffl:
#                 sleep_stage = ffl['stage'][()].flatten()
#                 start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signal_path,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [ ''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0]) ]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signal_path:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

#         # load labels
#         with h5py.File(label_path) as ffl:
#             sleep_stage = ffl['stage'][()].flatten()
#             if 'grass' in signal_path:
#                 start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
#             ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
#     if sleep_stage.shape[0]!=signal.shape[1]:
#         raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        signal_channel_names = channel_names
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        signal_channel_names = []
        for ichannel in ['ABD', 'CHEST']:
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        signal_channel_names = []
        for i in range(len(channels)):
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]#.T

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

#     # check whether sleep_stage contains all 5 stages
#     stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
#     if len(stages)<=1:
#         raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'channel_ids': signal_channel_ids, 'channel_names': signal_channel_names}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, params

In [4]:
import os
path_grass = '/media/mad3/Datasets_converteddata/sleeplab/grass_data/'
files_grass =[]
files_grass = [f for f in sorted(os.listdir(path_grass))]
files_grass = files_grass[1:]


path = '/media/mad3/Datasets_converteddata/sleeplab/grass_data/'
PLM = []
sleep = []

for k in range(1,2): # subject ID
    
    import datetime
    file_subject = files_grass [k]
    print ("subject = ", file_subject)

    Files = os.listdir(path + file_subject)
    LabelFile = [match for match in Files if "annotations" in match]
    SignalFile = [match for match in Files if "Signal" in match]

    
    if LabelFile and SignalFile: 
        
        signal_path = path + file_subject + '/' + SignalFile [0]
        label_path = path + file_subject + '/' + LabelFile [0]

        signal, params = check_load_mgh_dataset(signal_path)
        fs = params ['Fs']
        annotation = pd.read_csv(label_path)

        test = annotation.copy()
        start_time = annotation.iloc[0]['time']
        test = test[test['duration'].notna()]
        columnsName = test.event.unique().tolist()
        from datetime import datetime
        FMT = '%H:%M:%S'
        s1 = test.iloc[-1]['time']
        s2 = test.iloc[0]['time']
        tdelta = datetime.strptime(s1, FMT) - datetime.strptime(s2, FMT)
        len_in_seconds=tdelta.seconds
        label = pd.DataFrame(index=np.arange(len_in_seconds * int(fs)) , columns=columnsName)
        start_time = test.iloc [0]['time']
        start_time = datetime.strptime(start_time, '%H:%M:%S')
        for i in range(len(test)):
            current_time = test.iloc[i]['time']
            current_time = datetime.strptime(current_time, '%H:%M:%S')
            tdelta = time_diff(start_time , current_time) 
            len_in_seconds=tdelta.seconds
            time_samples_index = int (len_in_seconds * fs)
            duration_of_event = int (test.iloc [i]['duration'] * fs)
            if duration_of_event < 1:
                duration_of_event = int(np.ceil(duration_of_event))   
            event_name = test.iloc [i]['event']
            label.at [time_samples_index:(time_samples_index + duration_of_event), event_name] = 1

        mylist = label.columns.tolist()
        sub = 'lm'   
        PLM_events = [s for s in mylist if sub.lower() in s.lower()]
        PLM.append (PLM_events)
        
        se = 'Sleep'
        sleep_event = [s for s in mylist if se.lower() in s.lower()] 
        sleep.append(sleep_event)
        
        if signal.shape[1] > label.shape[0]:  # if length signal bigger than the events length in annoation 
            signal = signal[:, 0:label.shape[0]]
        else:   # if length signal smaller than the events length in annoation 
            N = label.shape[0] - signal.shape[1]  
            signal = np.pad(signal, (0, N), 'constant')
        
        
    PLM_values = annotation[annotation['event'].str.contains('PLM')]
    sleep_categorical = annotation[annotation['event'].str.contains('Sleep')]
    Sleep_values = annotation[annotation['event'].str.contains('Sleep')]

    event_interest = label [PLM_values.event.unique().tolist()]
    event_interest = event_interest.fillna(0)

    sleep_interest = label [Sleep_values.event.unique().tolist()]
    sleep_interest = sleep_interest.fillna(0)

    Sleep_values.event = pd.factorize(Sleep_values.event)[0]
    
    
    ch_name = ['F3-M2','F4-M1','C3-M2','C4-M1','O1-M2','O2-M1']

    channel_indexes = [params ['channel_names'].index(x) for x in ch_name if x in params ['channel_names']]
    interest_signal = signal [channel_indexes, :]



    # filtering signals 
    notch_freq = 60.  # [Hz]
    bandpass_freq = [0.1, 20]  # [Hz]
    highpass_freq = 10 # Hz for EMG
    epoch_time = 30   # [min]
    epoch_step_time = 30    # [min]
    std_thres = 0.2  # [uV]
    std_thres2 = 1.  # [uV]
    flat_seconds = 5  # [s]
    amplitude_thres = 500  # [uV]
    figsize = (18,5)
    eye_blinks = True
    Fs = params['Fs'] 
    Fs = int(Fs)

    Sleep_values = annotation[annotation['event'].str.contains('Sleep')]
    last_epoch = int(Sleep_values.iloc[-1]['epoch'])

    interest_signal = interest_signal[:, 0:last_epoch * epoch_time * Fs]


    if notch_freq is not None:
        psg = notch_filter(interest_signal, Fs, notch_freq, verbose=False)
    if bandpass_freq is not None:
        psg = filter_data(interest_signal, Fs, bandpass_freq[0], bandpass_freq[1], verbose=False)


    signals = np.transpose(psg).astype(np.float64)

    ##############################################

    # an anti-aliasing finite impulse response (FIR) filter is applied to all channels,
    # where the -3dB cut-off point is 28.29 Hz

    ##############################################
    # Apply antialiasing FIR filter to each channel and downsample to 50Hz

    filtCoeff = np.array([0.00637849379422531, 0.00543091599801427, -0.00255136650039784, -0.0123109503066702,
                          -0.0137267267561505, -0.000943230632358082, 0.0191919895027550, 0.0287148886882440,
                          0.0123598773891149, -0.0256928886371578, -0.0570987715759348, -0.0446385294777459,
                          0.0303553522906817, 0.148402006671856, 0.257171285176269, 0.301282456398562,
                          0.257171285176269, 0.148402006671856, 0.0303553522906817, -0.0446385294777459,
                          -0.0570987715759348, -0.0256928886371578, 0.0123598773891149, 0.0287148886882440,
                          0.0191919895027550, -0.000943230632358082, -0.0137267267561505, -0.0123109503066702,
                          -0.00255136650039784, 0.00543091599801427, 0.00637849379422531])

    for n in range(signals.shape[1]):
        signals[::, n] = np.convolve(signals[::, n], filtCoeff, mode='same')


    # # the channels are downsampled to 50 Hz
    # signals = signals[0::4,]

    garbageCollector.collect()

    # Normalize all the other channels by removing the mean and the rms in an 18 minute rolling window, 
    # using fftconvolve for computational efficiency 18 minute window is used because because baseline breathing
    # is established in 2 minute window according to AASM standards.
    # Normalizing over 18 minutes ensure a 90% overlap between the beginning and end of the baseline window

    kernel_size = (Fs*20*60)+1

    # Remove DC bias and scale for FFT convolution
    center = np.mean(signals, axis=0)
    scale = np.std(signals, axis=0)
    scale[scale == 0] = 1.0
    signals = (signals - center) / scale

    # Compute and remove moving average with FFT convolution
    center = np.zeros(signals.shape)
    for n in range(signals.shape[1]):
        center[::, n] = fftconvolve(signals[::, n], np.ones(shape=(kernel_size,))/kernel_size, mode='same')

    center[np.isnan(center) | np.isinf(center)] = 0.0
    signals = signals - center

    # Compute and remove the rms with FFT convolution of squared signal
    scale = np.ones(signals.shape)
    for n in range(signals.shape[1]):
        temp = fftconvolve(np.square(signals[::, n]), np.ones(shape=(kernel_size,))/kernel_size, mode='same')

        # Deal with negative values (mathematically, it should never be negative, but fft artifacts can cause this)
        temp[temp < 0] = 0.0

        # Deal with invalid values
        invalidIndices = np.isnan(temp) | np.isinf(temp)
        temp[invalidIndices] = 0.0
        maxTemp = np.max(temp)
        temp[invalidIndices] = maxTemp

        # Finish rms calculation
        scale[::, n] = np.sqrt(temp)


    scale[(scale == 0) | np.isinf(scale) | np.isnan(scale)] = 1.0  # To correct for record 12 that has a zero amplitude chest signal
    signals = signals / scale

    garbageCollector.collect()

    # N = 4 # downsample to 50 Hz
    # Fs = params['Fs'] 
    # Fs = int(Fs)
    # Fs = int( Fs/N )

    eeg_filtered = np.transpose(signals)

    epoch_size = int(round(epoch_time*Fs))
    epoch_step = int(round(epoch_step_time*Fs))
    start_ids = np.arange(0, eeg_filtered.shape[1]-epoch_size+1, epoch_step)
    seg_ids = list(map(lambda x:np.arange(x,x+epoch_size), start_ids))
    eeg_segs = eeg_filtered[:,seg_ids].transpose(1,0,2)  # eeg_segs.shape=(#epoch, #channel, Tepoch)


    # compute spectrogram
    import datetime
    from datetime import timedelta

    NW = 10.
    BW = NW*2./epoch_time
    specs, freq = psd_array_multitaper(eeg_segs, Fs, fmin=bandpass_freq[0], fmax=bandpass_freq[1], adaptive=False, low_bias=True, verbose='ERROR', bandwidth=BW, normalization='full')

    # mark artifact epochs
    flat_length = int(round(flat_seconds*Fs))
    epoch_state = ['good']*len(start_ids)

    # mark epochs with NaN
    nan2d = np.any(np.isnan(eeg_segs), axis=2)
    nan1d = np.where(np.any(nan2d, axis=1))[0]
    for i in nan1d:
        epoch_state[i] = 'EEG contains NaN'

    # mark large amplitude
    amplitude_large2d = np.any(np.abs(eeg_segs)>amplitude_thres, axis=2)
    amplitude_large1d = np.where(np.any(amplitude_large2d, axis=1))[0]
    for i in amplitude_large1d:
        epoch_state[i] = 'Large amplitude: higher than %guV'%amplitude_thres

    # mark flat signal with flat_length
    short_segs = eeg_segs.reshape(eeg_segs.shape[0], eeg_segs.shape[1], eeg_segs.shape[2]//flat_length, flat_length)
    flat2d = np.any(detrend(short_segs, axis=3).std(axis=3)<=std_thres, axis=2)
    flat2d = flat2d|(np.std(eeg_segs,axis=2)<=std_thres2)
    flat1d = np.where(np.any(flat2d, axis=1))[0]
    for i in flat1d:
        epoch_state[i] = 'Flat signal'
    print(Counter(epoch_state))

    # decide imshow vmax and vmin
    spec_db_vmin = []
    spec_db_vmax = []


    specs_db = 10*np.log10(specs)
    freq = freq.flatten()

    vmin, vmax = np.percentile(specs_db, (10,90))
    spec_db_vmin.append(vmin)
    spec_db_vmax.append(vmax)
    spec_db_vmin = min(spec_db_vmin)
    spec_db_vmax = max(spec_db_vmax)


    tt = np.arange(len(specs_db))*epoch_step_time/3600
    xticks = np.arange(0, np.floor(tt.max())+1) 
    xticklabels = []

    for j, x in enumerate(xticks):
        dt = start_time+timedelta(hours=x)
        #if j==0:
        xx = datetime.datetime.strftime(dt, '%H:%M:%S')#\n%m/%d/%Y')
        xticklabels.append(xx)

    vis_channel_names = ['Frontal','Central','Occipital']  # assume show 3 averaged channels
    plt.close()
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=len(vis_channel_names)+1, ncols=2, width_ratios=[50,1])


    ax = fig.add_subplot(gs[0,0])
    ax.step(np.arange(0,len(Sleep_values.time.tolist())), Sleep_values.event.tolist(), c='k')
    ax.set_ylim([0.5, 5.5])
    ax.set_xlim([0, len(Sleep_values)])
    ax.set_yticks([0,1,2,3,4,5])
    ax.set_yticklabels(['?', 'W', 'N1', 'N2', 'N3', 'R'])
    ax.set_ylabel('Sleep stage')
    ax.xaxis.set_major_formatter(plt.NullFormatter())
    locs = [i for i, e in enumerate(Sleep_values.time.tolist()) if e in xticklabels]
    ax.set_xticks(locs)


    for chi in range(len(vis_channel_names)):
        ax = fig.add_subplot(gs[chi+1,0])

        specs_db_ch = (specs_db[:,chi*2]+specs_db[:,chi*2+1])/2
        im = ax.imshow(specs_db_ch.T, cmap='jet', origin='lower', aspect='auto',
                vmax=spec_db_vmax ,vmin=spec_db_vmin,
                extent=(tt.min(),tt.max(),freq.min(),freq.max()))
        ax.text(-0.08, 0.5, 'Avg\n'+vis_channel_names[chi],
                    ha='center', va='center', transform=ax.transAxes)
        ax.set_ylabel('f (Hz)')
        ax.set_xticks(xticks)
        if chi==len(vis_channel_names)-1:
            ax.set_xlabel('time (h)')
            ax.set_xticklabels(xticklabels)
        else:
            ax.set_xticklabels([])
        if chi==len(vis_channel_names)-1:
            ax_cb = fig.add_subplot(gs[chi+1,1])
            fig.colorbar(im, cax=ax_cb, label='PSD (dB)')


    fig.suptitle(file_subject, fontsize=14)
    plt.tight_layout()
    plt.savefig('/home/snasiri/Dropbox (Partners HealthCare)/EMGs/plot_spectrogram/' + file_subject + '_EEG_Spectrogram.pdf')

subject =  ACHADINHA_KATARINA_021412_2222.000


/home/snasiri/.local/lib/python3.6/site-packages/pandas/core/generic.py:5168: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self[name] = value


Counter({'Flat signal': 830, 'good': 108})
